Первая имплементация не следует лучшим практикам. Там вызовы и получение данных зашиты внутрь состояния(State) агента. Это плохо, потому что данные с течением времени не меняются. Состояние должно содержать актуальные для работы агента данные, например вопросы от пользователей и доп контекст. Ещё одно проблема - слишком большая привязка к Langfuse. В новом подходе я сделаю отдельные шаги для моделей, которые будут получать нужные данные, а откуда они приходят модели знать не будут.

In [ ]:
import os
import random
from typing import Annotated, Any, TypedDict
import asyncio
from abc import ABC, abstractmethod

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import AIMessage
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langfuse.model import PromptClient
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import AnyMessage, add_messages
from pydantic import BaseModel, Field
from tqdm import tqdm
from langfuse._client.datasets import DatasetClient

load_dotenv()

os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-9058c552-1e8e-41c2-bcea-8abae84494da"
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-1434620a-ab84-4592-99f6-e3dd7419dd64"
os.environ["LANGFUSE_BASE_URL"] = "http://localhost:3000"

### Models

In [ ]:
class ActInfo(BaseModel):
    full_name: str | None = Field(
        default=None, description="Полное наименование правового акта (тип + наименование государственного органа)"
    )
    publication_date: str | None = Field(default=None, description="Дата опубликования документа")
    number: str | None = Field(default=None, description="Номер документа (например, 3789-р)")
    title: str | None = Field(default=None, description="Заголовок документа")
    government_agency_name: str | None = Field(default=None, description="Наименование государственного органа")
    signatory: str | None = Field(default=None, description="Официальное лицо, подписавшее акт")
    type: str | None = Field(default=None, description="Тип правового акта")

class RefineResponse(BaseModel):
    changes: str = Field(description="Какие были внесены изменения и почему")
    new_prompt: str = Field(description="Улучшенная версия промпта")

In [ ]:
generation_model = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    model_name=os.getenv("OPENAI_GENERATION_MODEL"),
    timeout=60,
    max_retries=3,
).with_structured_output(ActInfo)

refine_model = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    model_name=os.getenv("OPENAI_REFINE_MODEL"),
    timeout=60,
    max_retries=3,
).with_structured_output(RefineResponse, include_raw=True)


### Prompts v1

In [ ]:
GENERATION_PROMPT = """Извлеки данные из документа согласно заданной схеме.

# Документ
{{document}}"""

REFINE_PROMPT = """Проанализируй ответы модели по данным документам, подумай,
почему модель ошиблась и улучши промпт для генерации любыми известными техниками,
например, задание роли, добавление примеров и ограничений и другое. В ответе верни
улучшенную версию промпта и какие изменения ты внёс.

# Промпт модели
{{generation_prompt}}


# Схема данных модели
{{json_schema}}


# Примеры данных
{{examples}}
"""

class BasePromptClient(ABC):
    """Класс подставляет в промпт документы или любую
    другую информацию. Нужен для совместимости с Langfuse.
    Например, промпт как обычная строка будет выглядеть так:  
        "Извлеки информацию из документа: {document}"  
    Если промпт сохраняется в Langfuse, то у него свой шаблон подстановки:  
        "Извлеки информацию из документа: {{document}}"
        """
    def __init__(self, prompt: Any) -> None:
        self.prompt = prompt

    @abstractmethod
    def compile(self, **kwargs) -> str:
        """Сбор промпта и подстановка аргументов,
        если для них есть плейсхолдеры."""

class StrPromptClient(BasePromptClient):
    def compile(self, **kwargs):
        return self.prompt.format(**kwargs)
    
class LangfusePromptClient(BasePromptClient):
    def compile(self, **kwargs):
        return self.prompt.compile(**kwargs)

In [ ]:
class ResultScore(TypedDict):
    """Структура для сохранения плохих ответов маленькой модельки.
    В поля reference_answers и generated_answers должны сохраняться
    только примеры, где модель ошиблась. Если качество 1(100%), то
    эти поля будут пустыми.
    
    text: документ, который маленькая модель структурировала.
    score: оценка качества.
    reference_answers: ожидаемые ответы в формате поле: данные.
    generated_answers: сгенерированные ответы в формате поле: данные."""
    
    text: str
    score: float
    reference_answers: dict[str, Any]
    generated_answers: dict[str, Any]


# class State(TypedDict):
#     messages: Annotated[list[AnyMessage], add_messages]
#     repeat: int
#     current_step: int
#     bad_results: list[ResultScore]
#     best_test_accuracy: float
#     is_test_accuracy_improving: bool
#     best_generation_prompt_version: int
#     generation_prompt_client: PromptClient
#     refine_prompt_client: PromptClient

### Refine logic

In [ ]:
async def process_documents(
    generation_model: ChatOpenAI,
    documents: list[str],
    prompt: BasePromptClient,
) -> list[AIMessage]:

    tasks = []

    for doc in documents:
        doc = prompt.compile(document=doc)
        tasks.append(generation_model.ainvoke(doc))

    return asyncio.gather(*tasks)


def get_result_scores(
    documents: list[str],
    model_outputs: list[dict], 
    expected_outputs: list[dict],
) -> list[ResultScore]:
    #TODO docs
    
    result_scores = []

    for doc, mo, eo in zip(documents, model_outputs, expected_outputs):
        result_score: ResultScore = {}
        total_fields = len(mo)

        mo = set(mo.items())
        eo = set(eo.items())

        generated_answers = mo.difference(eo)
        reference_answers = eo.difference(mo)

        if generated_answers and reference_answers:
            result_score["generated_answers"] = dict(generated_answers)
            result_score["reference_answers"] = dict(reference_answers)
        
        result_score["text"] = doc
        result_score["score"] = 1 - len(generated_answers) / total_fields
    
        result_scores.append(result_score)
    
    return result_scores

def generate_bad_examples(result_scores: list[ResultScore]) -> str:

    bad_examples = ""
    results = [r for r in result_scores if r["score"] < 1]
    results = sorted(results, key=lambda x: x["score"])

    for result in results[:5]:
        bad_examples += f"Документ:\n{result["text"]}\n"
        reference = result["reference_answers"]
        generated = result["generated_answers"]
        for k in reference:
            bad_examples += f"Поле: {k}\nПравильный ответ: {reference[k]}\nОтвет модели: {generated[k]}\n"
        bad_examples += "\n"

    return bad_examples

async def refine_prompt(
    refine_model: ChatOpenAI,
    refine_prompt: BasePromptClient,
    generation_prompt: BasePromptClient,
    generation_model_structured_output: BaseModel,
    examples: str,
) -> AIMessage:

    prompt = refine_prompt.compile(
        generation_prompt=generation_prompt.compile(document="..."),
        json_schema=generation_model_structured_output.model_json_schema(),
        examples=examples,
    )

    output = await refine_model.ainvoke(prompt)
    return output

async def get_data_from_langfuse(dataset_client: DatasetClient, n_samples: int = 10, shuffle: bool = True):
    items = dataset_client.items

In [1]:
import random

In [ ]:
items = [1,2,3,4,5]

### Tests

In [ ]:
def test_get_result_scores_perfect():
    documents = ["doc1"]
    model_outputs = [{"a": 1, "b": 2, "c": 3}]
    expected_outputs = [{"a": 1, "b": 2, "c": 3}]
    result_scores = get_result_scores(documents, model_outputs, expected_outputs)
    assert result_scores[0]["score"] == 1.0

def test_get_result_scores_with_bad_result():
    documents = ["doc1"]
    model_outputs = [{"a": 1, "b": 2, "c": 3}]
    expected_outputs = [{"a": 1, "b": 2, "c": 2}]
    result_scores = get_result_scores(documents, model_outputs, expected_outputs)
    assert round(result_scores[0]["score"], 2) == 0.67

def test_get_result_scores_saves_bad_fields():
    documents = ["doc1"]
    model_outputs = [{"a": 1, "b": 2, "c": 3}]
    expected_outputs = [{"a": 1, "b": 2, "c": 2}]
    result_scores = get_result_scores(documents, model_outputs, expected_outputs)
    assert "c" in result_scores[0]["reference_answers"]
    assert "c" in result_scores[0]["generated_answers"]

def test_get_result_scores_all_bad():
    documents = ["doc1"]
    model_outputs = [{"a": 1, "b": 2, "c": 3}]
    expected_outputs = [{"a": 0, "b": 0, "c": 0}]
    result_scores = get_result_scores(documents, model_outputs, expected_outputs)
    assert result_scores[0]["score"] == 0.0

def test_get_result_scores_two_docs():
    documents = ["doc1", "doc2"]
    model_outputs = [{"a": 1, "b": 2, "c": 3}, {"a": 1, "b": 2, "c": 3}]
    expected_outputs = [{"a": 0, "b": 0, "c": 0}, {"a": 1, "b": 2, "c": 3}]
    result_scores = get_result_scores(documents, model_outputs, expected_outputs)

    assert result_scores[0]["text"] == "doc1"
    assert result_scores[0]["score"] == 0.0
    assert result_scores[1]["text"] == "doc2"
    assert result_scores[1]["score"] == 1.0


test_get_result_scores_perfect()
test_get_result_scores_with_bad_result()
test_get_result_scores_saves_bad_fields()
test_get_result_scores_all_bad()
test_get_result_scores_two_docs()